***

*Course:* [Math 535](https://people.math.wisc.edu/~roch/mmids/) - Mathematical Methods in Data Science (MMiDS)  
*Chapter:* 1-Introduction   
*Author:* [Sebastien Roch](https://people.math.wisc.edu/~roch/), Department of Mathematics, University of Wisconsin-Madison  
*Updated:* Jan 25, 2024  
*Copyright:* &copy; 2024 Sebastien Roch

***

In [ ]:
# FOR e-BOOK ONLY
import os, sys
sys.path.insert(0, os.path.abspath('../../utils')) # use directory to mmids.py

In [ ]:
# PYTHON 3
import numpy as np
from numpy import linalg as LA
import matplotlib.pyplot as plt
import pandas as pd
import mmids
seed = 535
rng = np.random.default_rng(seed)
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# FOR TeX ONLY
#plt.rcParams['figure.figsize'] = [4.00, 2.25] # for high-def figs
#plt.rcParams['figure.dpi'] = 200 # for high-def figs

## Some observations about high-dimensional data

In this section, we first apply $k$-means clustering to a high-dimensional example to illustrate the issues that arise in that context. We then discuss some surprising phenomena in high dimensions.

### Clustering in high dimension

In this section, we test our implementation of $k$-means on a simple simulated dataset in high dimension.

The following function generates $n$ data points from a mixture of two equally likely, spherical $d$-dimensional Gaussians with variance $1$, one with mean $-w\mathbf{e}_1$ and one with mean $w \mathbf{e}_1$. We use `gmm2` from a previous section. It is found in `mmids.py`, which can be downloaded [here](https://raw.githubusercontent.com/MMiDS-textbook/MMiDS-textbook.github.io/main/utils/mmids.py).

In [ ]:
def two_mixed_clusters(d, n, w):
    
    # set parameters
    phi1 = 0.5
    phi2 = 0.5
    mu1 = np.concatenate(([w], np.zeros(d-1)))
    mu2 = np.concatenate(([-w], np.zeros(d-1)))
    sigma1 = np.identity(d)
    sigma2 = np.identity(d)
    
    return mmids.gmm2(d, n, phi1, phi2, mu1, sigma1, mu2, sigma2)

**Two dimensions** We start with $d=2$.

In [ ]:
d, n, w = 2, 100, 3.
X = two_mixed_clusters(d, n, w)

We use a scatterplot to visualize the data. Each dot corresponds to one data point. Observe the two clearly delineated clusters.

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111,aspect='equal')
ax.scatter(X[:,0], X[:,1], s=10)
plt.show()

Let's run $k$-means on this dataset using $k=2$. We use `kmeans()` from the `mmids.py` file.

In [ ]:
assign = mmids.kmeans(X, 2)

Our default of $10$ iterations seem to have been enough for the algorithm to converge. We can visualize the result by [coloring](https://matplotlib.org/stable/api/_as_gen/matplotlib.lines.Line2D.html) the points according to the assignment.  

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111,aspect='equal')
ax.scatter(X[:,0], X[:,1], c=assign, s=10)
plt.show()

**General dimension** Let's see what happens in higher dimension. We repeat our experiment with $d=1000$.

In [ ]:
d, n, w = 1000, 100, 3.
X = two_mixed_clusters(d, n, w)

Again, we observe two clearly delineated clusters.

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111,aspect='equal')
ax.scatter(X[:,0], X[:,1], s=10)
plt.show()

This dataset is in $1000$ dimensions, but we've plotted the data in only the first two dimensions. If we plot in any two dimensions not including the first one instead, we see only one cluster. 

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111,aspect='equal')
ax.scatter(X[:,1], X[:,2], s=10)
plt.show()

Let's see how $k$-means fares on this dataset.

In [ ]:
assign = mmids.kmeans(X, 2)

Our attempt at clustering does not appear to have been successful.

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111,aspect='equal')
ax.scatter(X[:,0], X[:,1], c=assign, s=10)
plt.show()

Our attempt at clustering does not appear to have been successful. What happened? While these clusters are easy to tease apart *if we know to look at the first coordinate only*, in the full space the within-cluster and between-cluster distances become harder to distinguish: the noise overwhelms the signal. 

The function below plots the histograms of within-cluster and between-cluster distances for a sample of size $n$ in $d$ dimensions with a given offset. As $d$ increases, the two distributions become increasingly indistinguishable. Later in the course, we will develop dimension-reduction techniques that help deal with this issue. This time, we need labeled data points, i.e., we need to know the component from which each data point comes from. For this purpose, we implement a variant of our previous mixture simulator. This one generates -- separately -- each Gaussian and concatenates the data. So the first `n` samples are from one component and the next `n` samples are from the other.

In [ ]:
def one_cluster(d, n, w):
    X = np.stack(
        [np.concatenate(([w], np.zeros(d-1))) + rng.normal(0,1,d) for _ in range(n)]
    )
    return X

def two_clusters(d, n, w):
    X1 = one_cluster(d, n, -w)
    X2 = one_cluster(d, n, w)
    return X1, X2

In [ ]:
def highdim_2clusters(d, n, w):
    # generate datasets
    X1, X2 = two_clusters(d, n, w)
    
    # within-cluster distances for X1
    intra = np.stack(
        [LA.norm(X1[i,:] - X1[j,:]) for i in range(n) for j in range(n) if j>i]
    )
    plt.hist(intra, density=True, label='within-cluster')
    plt.title(f'dim={d}')
 
    # between-cluster distances
    inter = np.stack(
        [LA.norm(X1[i,:] - X2[j,:]) for i in range(n) for j in range(n)]
    )
    plt.hist(inter, density=True, alpha=0.75, label='between-cluster')

    plt.legend(loc='upper right')
    plt.title(f'dim={d}')

Next we plot the results for dimensions $d=2, 100, 1000$. What do you observe?

In [ ]:
highdim_2clusters(2, 100, 3)

In [ ]:
highdim_2clusters(100, 100, 3)

In [ ]:
highdim_2clusters(1000, 100, 3)

As the dimension increases, the distributions of intra-cluster and inter-cluster distances overlap significantly and become more or less indistinguishable. That provides some insights into why clustering may fail here. Note that we used the same offset for all simulations. On the other hand, if the separation between the clusters is sufficiently large, one would expect clustering to work even in high dimension. 

**TRY IT!** What precedes (and what follows in the next subsection) is not a formal proof that $k$-means clustering will be unsuccessful here. The behavior of the algorithm is quite complex and depends, in particular, on the initialization and the density of points. Here, increasing the number of data points eventually leads to a much better performance. Explore this behavior on your own by modifying the code. (For some theoretical justifications (beyond this course), see [here](https://arxiv.org/pdf/0912.0086.pdf) and [here](http://www.stat.yale.edu/~pollard/Papers/Pollard81AS.pdf).)

### Surprising phenomena in high dimension

<blockquote class="twitter-tweet" data-lang="en"><p lang="en" dir="ltr">a high-dimensional space is a lonely place</p>&mdash; Bernhard Schölkopf (@bschoelkopf) <a href="https://twitter.com/bschoelkopf/status/503554842829549568?ref_src=twsrc%5Etfw">August 24, 2014</a></blockquote> <script async src="https://platform.twitter.com/widgets.js" charset="utf-8"></script> 

In the previous section, we saw how the contribution from a large number of "noisy dimensions" can overwhelm the "signal" in the context of clustering. In this section we discuss further properties of high-dimensional space that are relevant to data science problems.  

Applying *Chebyshev's inequality* to sums of independent random variables has useful statistical implications: it shows that, with a large enough number of samples $n$, the sample mean is close to the population mean. Hence it allows us to infer properties of a population from samples. Interestingly, one can apply a similar argument to a different asymptotic regime: the limit of large dimension $d$. But as we will see in this section, the statistical implications are quite different.

**High-dimensional cube** To start explaining the quote above, we consider a simple experiment. Let $\mathcal{C} = [-1/2,1/2]^d$ be the $d$-cube with side lengths $1$ centered at the origin and let $\mathcal{B} = \{\mathbf{x} \in \mathbb{R}^d : \|\mathbf{x}\|\leq 1/2\}$ be the inscribed $d$-ball. 

Now pick a point $\mathbf{X}$ uniformly at random in $\mathcal{C}$. What is the probability that it falls in $\mathcal{B}$? 

To generate $\mathbf{X}$, we pick $d$ independent random variables $X_1, \ldots, X_d \sim \mathrm{U}[-1/2, 1/2]$, and form the vector $\mathbf{X} = (X_1, \ldots, X_d)$. Indeed, the PDF of $\mathbf{X}$ is then $f_{\mathbf{X}}(\mathbf{x})= 1^d = 1$ if $\mathbf{x} \in \mathcal{C}$ and $0$ otherwise.

The event we are interested in is $A = \left\{\|\mathbf{X}\| \leq 1/2\right\}$. The uniform distribution over the set $\mathcal{C}$ has the property that $\mathbb{P}[A]$ is the volume of $\mathcal{B}$ divided by the volume of $\mathcal{C}$. In this case, the volume of $\mathcal{C}$ is $1^d = 1$ and the volume of $\mathcal{B}$ has an [explicit formula](https://en.wikipedia.org/wiki/Volume_of_an_n-ball).

This leads to the following surprising fact:

**THEOREM** **(High-dimensional Cube)** Let 
$\mathcal{B} = \{\mathbf{x} \in \mathbb{R}^d \,:\, \|\mathbf{x}\|\leq 1/2\}$ and
$\mathcal{C} = [-1/2,1/2]^d$. Pick $\mathbf{X} \sim \mathrm{U}[\mathcal{C}]$. Then, as $d \to +\infty$,

$$
\mathbb{P}[\mathbf{X} \in \mathcal{B}]
\to 0.
$$

$\sharp$

In words, in high dimension if one picks a point at random from the cube, it is unlikely to be close to the origin. Instead it is likely to be in the corners. A geometric interpretation is that a high-dimensional cube is a bit like a "spiky ball." 

<!--SLIDES

<div>
<img src="./figs/ball_with_spikes.png" width="500"/>
</div>

**Figure:** Vizualization of a high-dimensional cube as a spiky ball (Credit: Made with Midjourney)
-->

**Figure:** Visualization of a high-dimensional cube as a spiky ball (*Credit:* Made with [Midjourney](https://www.midjourney.com/))

![Visualization of a high-dimensional cube](./figs/ball_with_spikes.png)

$\bowtie$

<!--TeX
![Vizualization of a high-dimensional cube as a spiky ball](./figs/ball_with_spikes.png)
-->

We give a proof based on *Chebyshev's inequality*. It has the advantage of providing some insight into this counter-intuitive phenomenon by linking it to the concentration of sums of independent random variables, in this case the squared norm of $\mathbf{X}$.

*Proof idea:* We think of $\|\mathbf{X}\|^2$ as a sum of independent random variables and apply *Chebyshev's inequality*. It implies that the norm of $\mathbf{X}$ is concentrated around its mean, which grows like $\sqrt{d}$. The latter is larger than $1/2$ for $d$ large.

*Proof:* To see the relevance of *Chebyshev's inequality*, we compute the mean and standard deviation of the norm of $\mathbf{X}$. In fact, because of the square root in $\|\mathbf{X}\|$, computing its expectation is difficult. Instead we work with the squared norm 

$$
\|\mathbf{X}\|^2 = X_1^2 + X_2^2 + \cdots + X_d^2,
$$

which has the advantage of being a sum of independent random variables - for which the expectation is much easier to compute. Observe further that the probability of the event of interest $\{\|\mathbf{X}\| \leq 1/2\}$ can be re-written in terms of $\|\mathbf{X}\|^2$ as follows

\begin{align*}
\mathbb{P}
\left[
\|\mathbf{X}\| \leq 1/2
\right]
&= 
\mathbb{P}
\left[
\|\mathbf{X}\|^2 \leq 1/4
\right].
\end{align*}

To simplify the notation, we use $\tilde\mu = \mathbb{E}[X_1^2]$ and $\tilde\sigma = \sqrt{\mathrm{Var}[X_1^2]}$ for the mean and standard deviation of $X_1^2$ respectively. Using linearity of expectation and the fact that the $X_i$'s are independent, we get

$$
\mu_{\|\mathbf{X}\|^2}
= \mathbb{E}\left[
\|\mathbf{X}\|^2
\right]
= \sum_{i=1}^d \mathbb{E}[X_i^2]
= d \,\mathbb{E}[X_1^2]
= \tilde\mu \, d,
$$

and

$$
\mathrm{Var}\left[
\|\mathbf{X}\|^2
\right]
= \sum_{i=1}^d \mathrm{Var}[X_i^2]
= d \,\mathrm{Var}[X_1^2].
$$

Taking a square root, we get an expression for the standard deviation of our quantity of interest $\|\mathbf{X}\|^2$ in terms of the standard deviation of $X_1^2$

$$
\sigma_{\|\mathbf{X}\|^2}
= \tilde\sigma \, \sqrt{d}.
$$
(Note that we could compute $\tilde\mu$ and $\tilde\sigma$ explicitly, but it will not be necessary here.)

We use *Chebyshev's inequality* to show that $\|\mathbf{X}\|^2$ is highly likely to be close to its mean $\tilde\mu \, d$, which is much larger than $1/4$ when $d$ is large. And that therefore $\|\mathbf{X}\|^2$ is highly unlikely to be smaller than $1/4$. We give the details next.

By the one-sided version of *Chebyshev's inequality* in terms of the standard deviation, we have

$$
\mathbb{P}\left[
\|\mathbf{X}\|^2 - \mu_{\|\mathbf{X}\|^2} \leq - \alpha
\right]
\leq 
\left(\frac{\sigma_{\|\mathbf{X}\|^2}}{\alpha}\right)^2.
$$

That is, using the formulas above and rearranging slightly,

$$
\mathbb{P}\left[
\|\mathbf{X}\|^2 \leq \tilde\mu \, d - \alpha
\right]
\leq 
\left(\frac{\tilde\sigma \, \sqrt{d}}{\alpha}\right)^2.
$$

How do we relate this to the probability of interest 
$\mathbb{P}\left[\|\mathbf{X}\|^2 \leq 1/4\right]$? Recall that we are free to choose $\alpha$ in this inequality. So simply take $\alpha$ such that

$$
\tilde\mu \,d - \alpha = \frac{1}{4},
$$

that is, $\alpha = \tilde\mu \,d - 1/4$. Observe that, once $d$ is large enough, it holds that $\alpha > 0$.

Finally, replacing this choice of $\alpha$ in the inequality above gives

\begin{align*}
\mathbb{P}
\left[
\|\mathbf{X}\| \leq 1/2
\right]
&= \mathbb{P}\left[\|\mathbf{X}\|^2 \leq 1/4\right]\\
&=
\mathbb{P}\left[
\|\mathbf{X}\|^2 \leq \tilde\mu \, d - \alpha
\right]\\
&\leq 
\left(\frac{\tilde\sigma \, \sqrt{d}}{\alpha}\right)^2\\
&\leq 
\left(\frac{\tilde\sigma \, \sqrt{d}}{\tilde\mu \,d - 1/4}\right)^2.
\end{align*}

Critically, $\tilde\mu$ and $\tilde\sigma$ do not depend on $d$. So the right-hand side goes to $0$ as $d \to +\infty$. Indeed, $d$ is much larger than $\sqrt{d}$ when $d$ is large. That proves the claim.$\square$

We will see later in the course that this high-dimensional phenomenon has implications for data science problems. It is behind what is referred to as the [Curse of Dimensionality](https://en.wikipedia.org/wiki/Curse_of_dimensionality).

While *Chebyshev's inequality* correctly implies that $\mathbb{P}[\mathbf{X} \in \mathcal{B}]$ goes to $0$, it does not give the correct rate of convergence. In reality, that probability goes to $0$ at a much faster rate than $1/d$. Specifically, [it can be shown](https://en.wikipedia.org/wiki/Volume_of_an_n-ball#High_dimensions) that $\mathbb{P}[\mathbf{X} \in \mathcal{B}]$ goes to $0$ roughly as $d^{-d/2}$. We will not need or derive this fact here.

**NUMERICAL CORNER:** We can check the theorem in a simulation. Here we pick $n$ points uniformly at random in the $d$-cube $\mathcal{C}$, for a range of dimensions up to `dmax`. We then plot the frequency of landing in the inscribed $d$-ball $\mathcal{B}$ and see that it rapidly converges to $0$. Alternatively, we could just plot the formula for the volume of $\mathcal{B}$. But knowing how to do simulations is useful in situations where explicit formulas are unavailable or intractable.

In [ ]:
def highdim_cube(dmax, n):
    
    in_ball = np.zeros(dmax)
    for d in range(dmax):
        # recall that d starts at 0 so we add 1 below
        in_ball[d] = np.mean(
            [(LA.norm(rng.random(d+1) - 1/2) < 1/2) for _ in range(n)]
        )
    
    plt.plot(np.arange(1,dmax+1), in_ball) 
    plt.xlabel('dim')
    plt.ylabel('in-ball freq')
    plt.show()

We plot the result up to dimension $10$.

In [ ]:
highdim_cube(10, 1000)

$\unlhd$

**Gaussians in high dimension** In this section, we turn our attention to the [Gaussian (or Normal) distribution](https://en.wikipedia.org/wiki/Normal_distribution) and its behavior in high dimension. 
Using *Chebyshev's inequality*, we show that a standard Normal vector has the following counter-intuititve property in high dimension: a typical draw has $2$-norm that is highly likely to be around $\sqrt{d}$. Visually, when $d$ is large, the joint PDF looks something like this:

**Figure:** A spherical shell ([Source](https://commons.wikimedia.org/wiki/File:Kugelschale.svg))

![A spherical shell](https://upload.wikimedia.org/wikipedia/commons/thumb/0/07/Kugelschale.svg/640px-Kugelschale.svg.png)

$\bowtie$

<!--TEX
![A spherical shell ([Source](https://commons.wikimedia.org/wiki/File:Kugelschale.svg))](./figs/2560px-Kugelschale.svg.png)
-->

This is unexpected because the joint PDF is maximized at $\mathbf{x} = \mathbf{0}$ for all $d$ (including $d=1$ as can be seen in the figure above). But the rough intuition is the following: (1) there is only "one way" to obtain $\|\mathbf{X}\|^2 = 0$ -- every coordinate must be $0$ by the point-separating property of the $2$-norm; (2) on the other hand, there are "many ways" to obtain $\|\mathbf{X}\|^2 = \sqrt{d}$ -- and that compensates for the lower density.

**THEOREM** **(High-dimensional Gaussians)** Let $\mathbf{X}$ be a standard Normal $d$-vector. Then, for any $\varepsilon > 0$,

$$
\mathbb{P}
\left[\,
\|\mathbf{X}\| \notin (\sqrt{d(1-\varepsilon)}, \sqrt{d(1+\varepsilon)})
\,\right]
\to 0
$$

as $d \to +\infty$. $\sharp$

*Proof idea:* We apply *Chebyshev's inequality* to the squared norm, which is a sum of independent random variables.

*Proof:* Let $Z = \|\mathbf{X}\|^2 = \sum_{i=1}^d X_i^2$ and notice that, by definition, it is a sum of independent random variables. Appealing to the expectation and variance formulas from the previous sections:

$$
\mathbb{E}[\|\mathbf{X}\|^2] = d \,\mathbb{E}[X_1^2] = d \,\mathrm{Var}[X_1] = d
$$

and

$$
\mathrm{Var}[\|\mathbf{X}\|^2] = d \,\mathrm{Var}[X_1^2]
$$

where $\mathrm{Var}[X_1^2]$ does not depend on $d$. By *Chebyshev's inequality*

$$
\mathbb{P}\left[
\|\mathbf{X}\|^2 \notin 
\left(
d(1-\varepsilon),d(1+\varepsilon)
\right)
\right]
=
\mathbb{P}[|\|\mathbf{X}\|^2 - d| \geq \varepsilon d]
\leq \frac{d \,\mathrm{Var}[X_1^2]}{\varepsilon^2 d^2}
= \frac{\mathrm{Var}[X_1^2]}{d \varepsilon^2}.
$$

Taking a square root inside the probability on the leftmost side and taking a limit as $d \to +\infty$ on the rightmost side gives the claim.$\square$

**NUMERICAL CORNER:** We check our claim in a simulation. We generate standard Normal $d$-vectors using the `rng.normal(0,1,d)` function and plot the histogram of their $2$-norm.

In [ ]:
def normal_shell(d, n):
    one_sample_norm = [LA.norm(rng.normal(0,1,d)) for _ in range(n)]
    plt.hist(one_sample_norm, bins=20)
    plt.xlim=(0,np.stack(one_sample_norm).max())
    plt.show()

We first plot it in one dimensions.

In [ ]:
normal_shell(1, 10000)

In higher dimension:

In [ ]:
normal_shell(100, 10000)

$\unlhd$